In [21]:
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import ast
from sklearn.metrics import f1_score, classification_report
from torch.amp import autocast, GradScaler
import random
import os
import numpy as np
import html
import re

In [22]:

def set_seed(seed=42):
    """Locks all random number generators for exact reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


set_seed(42)


In [23]:
 def clean_pcl_text(text):
    text = str(text)
    # Convert HTML entities back to normal characters
    text = html.unescape(text)
    # Strip out remaining HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Fix multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [24]:
# load data 
dev = 1
if dev ==1:
    # for dev set
    column_names = ['par_id', 'art_id', 'keyword', 'country_code', 'text', 'label']
    
    df_pcl = pd.read_csv('data/dontpatronizeme_pcl.tsv', 
                     sep='\t', 
                     skiprows=4, 
                     names=column_names, 
                     index_col=False,
                     quoting=3)
    df_dev_labels = pd.read_csv('data/dev_semeval_parids-labels.csv', index_col=False)
    
    df_pcl['pcl_presence'] = df_pcl['label'].apply(lambda x: 0 if x in [0, 1] else 1)
    
    import ast
    
    df_dev_labels['label'] = df_dev_labels['label'].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    df_pcl['text'] = df_pcl['text'].apply(clean_pcl_text)

    
        # Merge to get the text, keyword, and country code based on par_id
    df_test = pd.merge(df_dev_labels, df_pcl[['par_id', 'text', 'keyword', 'country_code',"pcl_presence"]], 
                         on='par_id', how='left')

       
        

else:

    # For test set
    test_file_path = "data/"
    df_test = pd.read_csv(test_file_path, sep='\t', header=None, 
                          names=['par_id', 'art_id', 'keyword', 'country_code', 'text'])

    df_test['text'] = df_test['text'].apply(clean_pcl_text)

print(df_test.head(1))


   par_id                  label  \
0    4046  [1, 0, 0, 1, 0, 0, 0]   

                                                text   keyword country_code  \
0  We also know that they can benefit by receivin...  hopeless           us   

   pcl_presence  
0             1  


In [25]:
# 3. Create the input string
df_test['model_input'] = (
    df_test['keyword'].astype(str) + " </s> " + 
    df_test['country_code'].astype(str) + " </s> " + 
    df_test['text'].astype(str)
)

# 4. Final preparation
if dev:
    df_test_prepared = df_test[['par_id', 'model_input', 'pcl_presence', 'label']].copy()
else:
    df_test_prepared = df_test[['par_id', 'model_input']].copy()

df_test_prepared = df_test_prepared.reset_index(drop=True)

In [26]:
model_name = 'roberta-large'
num_categories = 7


In [27]:
class PCLMultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256, has_labels=False):
        self.texts = df['model_input'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.has_labels = has_labels
        
        if self.has_labels:
            self.binary_labels = df['pcl_presence'].tolist()
            self.multi_labels = df['label'].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_length,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        if self.has_labels:
            item['binary_labels'] = torch.tensor([self.binary_labels[idx]], dtype=torch.float32)
            
            m_label = self.multi_labels[idx]
            if isinstance(m_label, str):
                m_label = ast.literal_eval(m_label)
            item['multi_labels'] = torch.tensor(m_label, dtype=torch.float32)

        return item

class MultiTaskRobertaPCL(nn.Module):
    def __init__(self, model_name=model_name, num_categories=num_categories):
        super(MultiTaskRobertaPCL, self).__init__()
        
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size 
        self.dropout = nn.Dropout(0.3)
    
        # 2 heads - one for binary classfication , another for categorical
        
        self.binary_head = nn.Linear(hidden_size, 1)
        self.category_head = nn.Linear(hidden_size, num_categories)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        
        # Extract the [CLS] token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        
        binary_logits = self.binary_head(pooled_output)
        category_logits = self.category_head(pooled_output)
        
        return binary_logits, category_logits


In [28]:
def run_inference(model, dataloader, device, threshold=0.5, has_labels=False):
    model.eval()
    dev_probs, dev_labels = [], []
    
    with torch.no_grad():
        for batch in dataloader:
            # 1. Forward pass
            outputs = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            
            # 2. Sigmoid on outputs[0] (binary_logits)
            probs = torch.sigmoid(outputs[0]) 
            
            # 3. Use extend just like your main code
            dev_probs.extend(probs.cpu().numpy())
            
            if has_labels:
                # Use .cpu() to ensure it works on both GPU and CPU
                dev_labels.extend(batch['binary_labels'].cpu().numpy())
                
    # 4. Flattening logic exactly like your main code
    y_dev_probs = np.array(dev_probs).flatten()
    
    # 5. FIXED: Using >= to match your main code logic
    final_dev_preds = (y_dev_probs >= threshold).astype(int)

    if has_labels:
        y_dev_true = np.array(dev_labels).flatten()
        
        # 6. Evaluation matching your main code
        final_dev_f1 = f1_score(y_dev_true, final_dev_preds, pos_label=1, zero_division=0)
        
        print(f"\nFinal F1 : {final_dev_f1:.4f}")
        print("\n--- FINAL CLASSIFICATION REPORT ---")
        print(classification_report(y_dev_true, final_dev_preds, target_names=['Non-PCL (0)', 'PCL (1)']))
        
        return final_dev_preds, y_dev_true
    
    return final_dev_preds

In [29]:

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_name = 'roberta-large'
    optimal_threshold = 0.89  
    if dev:
        
        has_labels = True
    else:
        has_labels = False

    # 1. Load Tokenizer & Model
    print("Loading model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = MultiTaskRobertaPCL(model_name=model_name)
    
    # LOAD YOUR SAVED WEIGHTS HERE
    model.load_state_dict(torch.load('models/best_roberta_large_model_v2.pt', map_location=device))
    model.to(device)

    # 2. Prepare DataLoaders
    # Assuming df_dev_prepared and df_test_prepared are already loaded/cleaned in memory
    print("Preparing DataLoaders...")

    test_dataset = PCLMultiTaskDataset(df_test_prepared, tokenizer, has_labels=has_labels)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

    # 4. Run Test Set (Generates Submission)
    print("\nRunning Official Test Set...")
    test_predictions = run_inference(
        model, test_loader, device, threshold=optimal_threshold, has_labels=has_labels
    )

    # # 5. Save final submission file
    if dev:
        
        with open('data/dev_submission.txt', 'w') as f:
            for pred in test_predictions:
                f.write(f"{pred}\n")
        print(f"\nSaved {len(test_predictions)} predictions to dev submission.txt")
    else:
        with open('data/test_submission.txt', 'w') as f:
            for pred in test_predictions:
                f.write(f"{pred}\n")
        print(f"\nSaved {len(test_predictions)} predictions to test submission.txt")
        
        

Loading model and tokenizer...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Preparing DataLoaders...

Running Official Test Set...

Final F1 : 0.6179

--- FINAL CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

 Non-PCL (0)       0.96      0.95      0.96      1895
     PCL (1)       0.58      0.66      0.62       199

    accuracy                           0.92      2094
   macro avg       0.77      0.80      0.79      2094
weighted avg       0.93      0.92      0.92      2094


Saved 2 predictions to dev submission.txt


In [30]:

# --- NEW CODE: STORE THE PREDICTIONS ---
output_file = "data/dev_submission.txt"

with open(output_file, 'w') as f:
    for pred in final_dev_preds:
        f.write(f"{pred}\n")

print(f"\nSaved {len(final_dev_preds)} predictions to '{output_file}'")

NameError: name 'final_dev_preds' is not defined